In [1]:
import time
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda,
    RunnableBranch,
)
from IPython.display import display, Markdown

# ============================================================
#  CONFIGURATION
# ============================================================
MODEL = "llama3.2:3b"

llm = ChatOllama(model=MODEL, temperature=0.7)

# Quick connectivity test
test = llm.invoke("Say 'ready' in one word.")
print(f"\u2705 Connected to Ollama | Model: {MODEL}")
print(f"   Response: {test.content}")

c:\Users\shiva\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


✅ Connected to Ollama | Model: llama3.2:3b
   Response: Ready.


---

## 1. Sequential Chains — Linear Pipelines

The most common pattern. Data flows through a linear pipeline where each step transforms it for the next.

```
Input → Step A → Step B → Step C → Output
```

### Experiment 1A: The Basic Chain — prompt | model | parser

In [2]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise technical assistant. Only answer in 2 sentences max."),
    ("human", "{question}")
])

parser = StrOutputParser()

llm = ChatOllama(model=MODEL, temperature=0.7)

basic_chain = prompt | llm | parser

result = basic_chain.invoke({"question": "What are chains in LangChain?"})
print(f"Type: {type(result).__name__}")  # str
display(Markdown(result))

Type: str


In LangChain, a "chain" refers to a sequence of language models or other components that are connected together to perform a specific task, such as text classification, question answering, or conversation generation.

A chain typically consists of multiple components, each of which takes the output of its predecessor and produces an input for the next component. This allows LangChain to build complex pipelines that can handle tasks with multiple steps or interactions.

### Experiment 1B: Multi-Step Sequential Chain

In [4]:
idea_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You generate creative app ideas. Respond with ONLY the app name and a 1-2 sentence description. No bullet points, no headers, no explanations."),
        ("human", "Give me an app idea for: {domain}")
    ])
    | llm | StrOutputParser()
)

In [5]:
tagline_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You write catchy taglines. Respond with ONLY one tagline sentence. Nothing else."),
        ("human", "Write a tagline for this app idea: {idea}")
    ])
    | llm | StrOutputParser()
)

In [6]:
features_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You list product features. Respond with ONLY 3 bullet points, one line each. No intro, no outro."),
        ("human", "List 3 key features for: {idea}")
    ])
    | llm | StrOutputParser()
)

In [7]:
domain = "education"

In [8]:
print(f"Domain: {domain}")
print(f"\n{'=' * 50}")
print("Step 1 — App Idea:")
print(f"{'=' * 50}")
idea = idea_chain.invoke({"domain": domain})
print(idea)

Domain: education

Step 1 — App Idea:
MindPal


In [9]:
print(f"\n{'=' * 50}")
print("Step 2 — Tagline:")
print(f"{'=' * 50}")
tagline = tagline_chain.invoke({"idea": idea})
print(tagline)


Step 2 — Tagline:
"Find your inner calm, one thought at a time."


In [10]:
print(f"\n{'=' * 50}")
print("Step 3 — Key Features:")
print(f"{'=' * 50}")
features = features_chain.invoke({"idea": idea})
display(Markdown(features))


Step 3 — Key Features:


• **Personalized AI Assistant**: MindPal is a personalized AI assistant that learns your habits, preferences, and goals to provide tailored suggestions and recommendations.
• **Mental Health Support**: The app offers mental health support through its chatbot feature, which uses natural language processing (NLP) and machine learning algorithms to detect emotions and provide personalized guidance.
• **Customizable Goal Setting**: MindPal allows users to set and track their personal goals, providing a structured approach to goal setting and progress tracking.

### Experiment 1C: Context Accumulation with `assign()`

In multi-step chains, later steps often need outputs from **earlier** steps — not just the previous one. `RunnablePassthrough.assign()` accumulates context so every step sees everything before it.

```
Start:        {"topic": "AI"}
After Step 1: {"topic": "AI", "summary": "..."}
After Step 2: {"topic": "AI", "summary": "...", "key_points": "..."}
After Step 3: {"topic": "AI", "summary": "...", "key_points": "...", "verdict": "..."}
```

In [25]:
summarize_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You write concise technical notes. "
         "Summarize the topic in plain prose. "
         "Your response must be under 40 words total."),
        ("human", "Summarize: {topic}")
    ])
    | llm | StrOutputParser()
)

In [ ]:
keypoints_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "Based on this summary: {summary}\n\n"
         "Extract 3 key points. One short line per point, under 10 words each.\n"
         "Use this exact format:\n"
         "1. First point here\n"
         "2. Second point here\n"
         "3. Third point here\n\n"
         "Write nothing else. Only 3 lines."),
        ("human", "Topic: {topic}")
    ])
    | llm | StrOutputParser()
)

In [27]:
verdict_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "Summary: {summary}\n"
         "Key Points: {key_points}\n\n"
         "Write a single-sentence verdict on why this topic matters. "
         "Your response must be under 25 words. No lists, no headings."),
        ("human", "Topic: {topic}")
    ])
    | llm | StrOutputParser()
)


In [28]:
analysis_pipeline = (
    RunnablePassthrough.assign(summary=summarize_chain) | RunnablePassthrough.assign(key_points=keypoints_chain) | RunnablePassthrough.assign(verdict=verdict_chain)
)

In [29]:
result = analysis_pipeline.invoke({"topic": """

Argentina has once again demonstrated why it remains one of the strongest football nations in the world during the 2026 FIFA World Cup. As the defending champions, the team entered the tournament with high expectations and has largely lived up to them through disciplined performances, tactical flexibility, and remarkable resilience. Under the guidance of Lionel Scaloni, Argentina successfully topped Group J by defeating Algeria, Austria, and Jordan, displaying a balanced style of play that combined solid defending with clinical finishing. The team relied on experienced leaders such as Lionel Messi while also benefiting from the energy and creativity of younger players like Enzo Fernández, Julián Álvarez, and Lautaro Martínez.

Argentina's knockout-stage journey has been particularly dramatic. In the Round of 32, they overcame a determined Cape Verde side in an exciting 3–2 victory, proving their ability to perform under pressure. Their Round of 16 encounter against Egypt became one of the most memorable matches of the tournament. Trailing 2–0 with only minutes remaining, Argentina produced an extraordinary comeback. Goals from Cristian Romero, Lionel Messi, and Enzo Fernández completed a stunning 3–2 victory, highlighting the team's fighting spirit and never-give-up attitude. Messi once again played a decisive role by inspiring the comeback, scoring a crucial goal, and leading the team with his experience and composure.

Despite these impressive victories, Argentina has also shown certain vulnerabilities, particularly in defense, having conceded multiple goals in consecutive knockout matches. However, their attacking quality, mental strength, and championship experience have enabled them to overcome difficult situations. As they prepare for the quarter-finals against Switzerland, Argentina remains one of the favorites to retain the World Cup title. Their blend of experienced stars and emerging talents, combined with a winning mentality, continues to make them one of the most exciting and dangerous teams in the competition.

 """})

In [30]:
print("PROGRESSIVE CONTEXT ACCUMULATION")
print("=" * 50)
print(f"\nTopic: {result['topic']}")
print(f"\nSummary:\n  {result['summary']}")
print(f"\nKey Points:\n  {result['key_points']}")
print(f"\nVerdict:\n  {result['verdict']}")

PROGRESSIVE CONTEXT ACCUMULATION

Topic: 

Argentina has once again demonstrated why it remains one of the strongest football nations in the world during the 2026 FIFA World Cup. As the defending champions, the team entered the tournament with high expectations and has largely lived up to them through disciplined performances, tactical flexibility, and remarkable resilience. Under the guidance of Lionel Scaloni, Argentina successfully topped Group J by defeating Algeria, Austria, and Jordan, displaying a balanced style of play that combined solid defending with clinical finishing. The team relied on experienced leaders such as Lionel Messi while also benefiting from the energy and creativity of younger players like Enzo Fernández, Julián Álvarez, and Lautaro Martínez.

Argentina's knockout-stage journey has been particularly dramatic. In the Round of 32, they overcame a determined Cape Verde side in an exciting 3–2 victory, proving their ability to perform under pressure. Their Round o

### Sequential PASSTHROUGH OUTPUT OF ONE TO ANOTHER

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

MODEL = "llama3.2:3b"
llm = ChatOllama(model=MODEL, temperature=0.2)
parser = StrOutputParser()

summarize_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a precise summarizer."),
        ("human",
         "Summarize the following topic in under 40 words.\n"
         "Output ONLY the summary text. No preamble, no 'Here is...', no labels.\n\n"
         "Topic:\n{topic}")
    ])
    | llm | parser
)

keypoints_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You extract key points. You never add preamble or commentary."),
        ("human",
         "Summary:\n{summary}\n\n"
         "Extract exactly 3 key points from this summary.\n"
         "Output ONLY in this format, nothing else:\n"
         "1. ...\n2. ...\n3. ...")
    ])
    | llm | parser
)

verdict_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You explain why a topic MATTERS — its significance, impact, or relevance. "
         "You do NOT summarize or restate facts already given."),
        ("human",
         "Summary:\n{summary}\n\n"
         "Key Points:\n{key_points}\n\n"
         "In ONE sentence (max 25 words), explain why this topic is significant or useful — "
         "not what it is. Output ONLY that sentence, no preamble.")
    ])
    | llm | parser
)

analysis_pipeline = (
    RunnablePassthrough.assign(summary=summarize_chain)
    .assign(key_points=keypoints_chain)
    .assign(verdict=verdict_chain)
)

result = analysis_pipeline.invoke({"topic": "..."})

print("Summary")
print(result["summary"])
print("keyPoints")
print(result["key_points"])
print("verdict")
print(result["verdict"])

Summary
Please provide the topic you'd like summarized. I'll be happy to assist!
keyPoints
Sentence transformer models are designed to transform sentences into numerical vectors for NLP tasks such as text classification and sentiment analysis.

They use a multi-layer bidirectional transformer encoder architecture to capture the semantic meaning of entire sentences rather than individual words.

Sentence transformers have several advantages over traditional word embeddings, including improved semantic understanding and increased representation capacity.
verdict
Capturing the semantic meaning of entire sentences enables more accurate and nuanced understanding of context in NLP tasks, improving overall model performance.


In [43]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

MODEL = "llama3.2:3b"
llm = ChatOllama(model=MODEL, temperature=0.2)
parser = StrOutputParser()

summarize_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a precise summarizer."),
        ("human",
         "Summarize the following topic in under 40 words.\n"
         "Output ONLY the summary text. No preamble, no 'Here is...', no labels.\n\n"
         "Topic:\n{topic}")
    ])
    | llm | parser
)

keypoints_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You extract key points. You never add preamble or commentary."),
        ("human",
         "Summary:\n{summary}\n\n"
         "Extract exactly 3 key points from this summary.\n"
         "Output ONLY in this format, nothing else:\n"
         "1. ...\n2. ...\n3. ...")
    ])
    | llm | parser
)

verdict_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You explain why a topic MATTERS — its significance, impact, or relevance. "
         "You do NOT summarize or restate facts already given."),
        ("human",
         "Summary:\n{summary}\n\n"
         "Key Points:\n{key_points}\n\n"
         "In ONE sentence (max 25 words), explain why this topic is significant or useful — "
         "not what it is. Output ONLY that sentence, no preamble.")
    ])
    | llm | parser
)

analysis_pipeline = (
    RunnablePassthrough.assign(summary=summarize_chain).assign(key_points=keypoints_chain).assign(verdict=verdict_chain)
)

result = analysis_pipeline.invoke({"topic": """
Argentina has once again demonstrated why it remains one of the strongest football nations in the world during the 2026 FIFA World Cup. As the defending champions, the team entered the tournament with high expectations and has largely lived up to them through disciplined performances, tactical flexibility, and remarkable resilience. Under the guidance of Lionel Scaloni, Argentina successfully topped Group J by defeating Algeria, Austria, and Jordan, displaying a balanced style of play that combined solid defending with clinical finishing. The team relied on experienced leaders such as Lionel Messi while also benefiting from the energy and creativity of younger players like Enzo Fernández, Julián Álvarez, and Lautaro Martínez.

Argentina's knockout-stage journey has been particularly dramatic. In the Round of 32, they overcame a determined Cape Verde side in an exciting 3–2 victory, proving their ability to perform under pressure. Their Round of 16 encounter against Egypt became one of the most memorable matches of the tournament. Trailing 2–0 with only minutes remaining, Argentina produced an extraordinary comeback. Goals from Cristian Romero, Lionel Messi, and Enzo Fernández completed a stunning 3–2 victory, highlighting the team's fighting spirit and never-give-up attitude. Messi once again played a decisive role by inspiring the comeback, scoring a crucial goal, and leading the team with his experience and composure.

Despite these impressive victories, Argentina has also shown certain vulnerabilities, particularly in defense, having conceded multiple goals in consecutive knockout matches. However, their attacking quality, mental strength, and championship experience have enabled them to overcome difficult situations. As they prepare for the quarter-finals against Switzerland, Argentina remains one of the favorites to retain the World Cup title. Their blend of experienced stars and emerging talents, combined with a winning mentality, continues to make them one of the most exciting and dangerous teams in the competition.

"""})

print("Summary")
print(result["summary"])
print("keyPoints")
print(result["key_points"])
print("verdict")
print(result["verdict"])

Summary
Argentina has lived up to high expectations at the 2026 FIFA World Cup, displaying disciplined performances, tactical flexibility, and resilience under Lionel Scaloni's guidance, with key players like Messi leading the team to dramatic comebacks.
keyPoints
1. Argentina has displayed disciplined performances at the 2026 FIFA World Cup.
2. The team has shown tactical flexibility under Lionel Scaloni's guidance.
3. Key players like Messi have led the team to dramatic comebacks.
verdict
Argentina's success at the 2026 FIFA World Cup highlights the effectiveness of Lionel Scaloni's management and the enduring impact of key players like Messi on team dynamics.


---

## 2. Parallel Chains — Execute Branches Simultaneously

Run multiple independent operations on the same input **at the same time**.

```
         ┌→ Branch A →┐
Input →  ├→ Branch B →├→ Merged Output
         └→ Branch C →┘
```

### Experiment 2A: Parallel Analysis — Multiple Aspects at Once

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama
from IPython.display import Markdown, display

MODEL = "llama3.2:3b"
llm = ChatOllama(model=MODEL, temperature=0.1)
parser = StrOutputParser()

# -----------------------------------------
# All three run in parallel, off raw topic
# -----------------------------------------
summary_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You write concise technical summaries. "
         "Summarize in plain prose. Under 30 words. "
         "Output ONLY the summary text, no preamble."),
        ("human", "{topic}")
    ])
    | llm
    | parser
)

pros_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "List exactly 2 advantages of the given topic. "
         "One line each, under 12 words per line.\n"
         "Format:\n- Advantage one\n- Advantage two\n"
         "Write nothing else."),
        ("human", "{topic}")
    ])
    | llm
    | parser
)

cons_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "List exactly 2 limitations of the given topic. "
         "One line each, under 12 words per line.\n"
         "Format:\n- Limitation one\n- Limitation two\n"
         "Write nothing else."),
        ("human", "{topic}")
    ])
    | llm
    | parser
)

analysis_pipeline = RunnableParallel(
    summary=summary_chain,
    pros=pros_chain,
    cons=cons_chain,
)

c:\Users\shiva\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
result = analysis_pipeline.invoke({"topic": "Using chains in LangChain for production apps"})

In [4]:
print("SUMMARY:")
print(f"  {result['summary']}")
print(f"\nPROS:")
print(f"  {result['pros']}")
print(f"\nCONS:")
print(f"  {result['cons']}")

SUMMARY:
  Chains in LangChain enable the creation of complex workflows by chaining together multiple tasks, allowing developers to build scalable and maintainable applications.

To use chains in a production app:

1. Define tasks: Break down your workflow into individual tasks that can be executed sequentially or concurrently.
2. Create a chain: Use LangChain's `Chain` class to create a new chain, specifying the order of execution for each task.
3. Add tasks to the chain: Use the `add_task` method to add tasks to the chain, specifying any necessary dependencies between tasks.
4. Handle errors and exceptions: Implement error handling mechanisms to catch and handle any errors that may occur during task execution.
5. Test and iterate: Thoroughly test your chain and iterate on its design as needed to ensure it meets your application's requirements.

Benefits of using chains in production apps:

* **Modularity**: Chains enable modular development, allowing developers to focus on individual

In [5]:
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.runnables import RunnableParallel
# from langchain_core.output_parsers import StrOutputParser
# from langchain_ollama import ChatOllama
# from IPython.display import Markdown, display

# MODEL = "llama3.2:3b"
# llm = ChatOllama(model=MODEL, temperature=0.1)
# parser = StrOutputParser()

# # -----------------------------------------
# # All three run in parallel, off raw topic
# # -----------------------------------------
# summary_chain = (
#     ChatPromptTemplate.from_messages([
#         ("system",
#          "You write concise technical summaries. "
#          "Summarize in plain prose. Under 30 words. "
#          "Output ONLY the summary text, no preamble."),
#         ("human", "{topic}")
#     ])
#     | llm
#     | parser
# )

# pros_chain = (
#     ChatPromptTemplate.from_messages([
#         ("system",
#          "List exactly 2 advantages of the given topic. "
#          "One line each, under 12 words per line.\n"
#          "Format:\n- Advantage one\n- Advantage two\n"
#          "Write nothing else."),
#         ("human", "{topic}")
#     ])
#     | llm
#     | parser
# )

# cons_chain = (
#     ChatPromptTemplate.from_messages([
#         ("system",
#          "List exactly 2 limitations of the given topic. "
#          "One line each, under 12 words per line.\n"
#          "Format:\n- Limitation one\n- Limitation two\n"
#          "Write nothing else."),
#         ("human", "{topic}")
#     ])
#     | llm
#     | parser
# )

# analysis_pipeline = RunnableParallel(
#     summary=summary_chain,
#     pros=pros_chain,
#     cons=cons_chain,
# )

# -----------------------------------------
# Synthesis (needs summary, pros, cons only)
# -----------------------------------------
synthesis_prompt = ChatPromptTemplate.from_messages([
    ("system", "You write balanced technical assessments."),
    ("human",
     "Summary: {summary}\n"
     "Pros: {pros}\n"
     "Cons: {cons}\n\n"
     "Write a brief assessment in plain prose, under 60 words. "
     "No bullet points, no headings, no preamble.")
])

full_analysis = analysis_pipeline | synthesis_prompt | llm | parser

# -----------------------------------------
# Run
# -----------------------------------------
result = full_analysis.invoke({"topic": "Using chains in LangChain for production apps"})

print("SYNTHESIZED ASSESSMENT:")
print("=" * 50)
display(Markdown(result))

SYNTHESIZED ASSESSMENT:


Chains in LangChain enable modular and scalable application development by breaking down complex workflows into smaller tasks that can be executed sequentially. This approach promotes flexibility and customizability through event handling and error handling, making it suitable for data processing pipelines, API workflows, and machine learning workflows.

---

## 3. Conditional Chains — Routing by Condition

Route input to different processing paths based on conditions.

```
Input → Condition Check
            ├→ True  → Chain A
            └→ False → Chain B
```

### Experiment 3A: Keyword-Based Routing with RunnableBranch

In [9]:
from langchain_core.runnables import RunnableLambda, RunnableBranch

In [10]:
technical_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You are a precise software engineer writing quick technical answers. "
         "Use technical terms. Your response must be under 40 words."),
        ("human", "{question}")
    ]) | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[TECHNICAL] {x}")
)

simple_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You explain things to a 10-year-old using a single analogy. "
         "Your response must be under 40 words."),
        ("human", "{question}")
    ]) | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[SIMPLE] {x}")
)

general_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You are a helpful assistant who gives short, direct answers. "
         "Your response must be under 40 words."),
        ("human", "{question}")
    ]) | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[GENERAL] {x}")
)

router = RunnableBranch(
    (lambda x: any(w in x["question"].lower() for w in ["code", "api", "implement", "function"]),
     technical_chain),
    (lambda x: any(w in x["question"].lower() for w in ["simple", "easy", "explain like", "beginner"]),
     simple_chain),
    general_chain  # default
)

In [11]:
test_questions = [
    "How do I implement a sequential chain in code?",
    "Explain chains in a simple way for a beginner.",
    "What are the benefits of using LangChain?",
]

In [12]:
for q in test_questions:
    print(f"\n{'=' * 50}")
    print(f"Q: {q}")
    print(f"{'=' * 50}")
    print(router.invoke({"question": q}))


Q: How do I implement a sequential chain in code?
[TECHNICAL] Here's an example of implementing a sequential chain using a simple linked list data structure in Python:

```python
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None

class SequentialChain:
    def __init__(self):
        self.head = None

    def append(self, value):
        new_node = Node(value)
        if not self.head:
            self.head = new_node
        else:
            current = self.head
            while current.next:
                current = current.next
            current.next = new_node

    def traverse(self):
        values = []
        current = self.head
        while current:
            values.append(current.value)
            current = current.next
        return values

# Example usage:
chain = SequentialChain()
chain.append(1)
chain.append(2)
chain.append(3)

print(chain.traverse())  # Output: [1, 2, 3]
```

In this example:

*   We define a `Node` c

In [14]:
from langchain_core.runnables import RunnablePassthrough

### Experiment 3B: LLM-Based Routing (Classify → Route)

Instead of hardcoded keyword rules, use the **LLM itself** to classify and route.

In [37]:
classifier = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You are a question classifier. "
         "Classify this question into exactly one category. Be clear about the classification when related to coding langugaes or questions on computer based application classify as technical, creative metaphor and analogies as creative and when it is not explicitly both of these, give the output as general\n"
         "Valid categories: technical, creative, general\n"
         "Respond with ONLY that one word. Nothing else."),
        ("human", "{question}")
    ]) | llm | StrOutputParser()
)

classified = RunnablePassthrough.assign(category=classifier)

In [38]:
technical_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You are a precise software engineer writing quick technical answers and coding related questions include chain building also "
         "Use technical terms. Your response must be under 40 words."),
        ("human", "{question}")
    ]) | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[TECHNICAL] {x}")
)


general_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You are a helpful assistant who gives short, direct answers. "
         "Your response must be under 40 words."),
        ("human", "{question}")
    ]) | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[GENERAL] {x}")
)


creative_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You are a creative storyteller. Use vivid imagery and metaphor. "
         "Your response must be under 40 words."),
        ("human", "{question}")
    ]) | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[CREATIVE] {x}")
)

In [39]:
smart_router = RunnableBranch(
    (lambda x: "technical" in x["category"].lower(), technical_chain),
    (lambda x: "creative" in x["category"].lower(), creative_chain),
    general_chain
)

In [40]:
smart_pipeline = classified | smart_router

In [41]:
test_qs = [
    "Tell me about the advantages of python coding language",
    "Write a metaphor about data pipelines.",
    "What is the capital of France?",
]


In [42]:
for q in test_qs:
    result = smart_pipeline.invoke({"question":test_qs[0]})


In [43]:
print(result)

[TECHNICAL] Python is a high-level, interpreted programming language that offers numerous advantages:

1. **Easy to Learn**: Python has a simple syntax and is relatively easy to learn, making it an ideal language for beginners.
2. **Fast Development**: Python's syntax and nature enable rapid development and prototyping, allowing developers to quickly test and iterate on their ideas.
3. **Large Standard Library**: Python comes with a vast collection of libraries and modules that make it easy to perform various tasks, such as data analysis, web development, and more.
4. **Cross-Platform**: Python can run on multiple operating systems, including Windows, macOS, and Linux.
5. **Dynamic Typing**: Python's dynamic typing system allows for flexibility in coding, making it easier to write code that is adaptable to changing requirements.
6. **Extensive Community**: Python has a massive and active community of developers, which means there are many resources available for learning and troublesho